In [83]:
import pandas as pd

df = pd.read_pickle("scraped_pages.pkl").reset_index(drop=True)
df.head()

,Unnamed: 0,Code,Code.1,Description,Type,Unit\n,Rate £,Extracted Price,New Rate (ex VAT),New Rate INCL VAT),...,Supplier Code,Supplier,URL,Comments,Source\n,Web,Status,Unnamed: 21,InApp Comment,html_source
0,NaN,10537104,10537104,Fusible Link Fire Damper with installation fra...,1,each,86.91,NaN,NaN,NaN,...,CFDIN,DUCTSTORE.CO.UK,https://www.ductstore.co.uk/cgi-bin/sh000055.p...,March 24 ce price updates prices shown ex vat ...,ductstore.co,Yes,NaN,NaN,NaN,"<!DOCTYPE html PUBLIC ""-//W3C//DTD HTML 4.01 T..."
1,NaN,10537106,10537106,Fusible Link Fire Damper with installation fra...,1,each,86.91,NaN,NaN,NaN,...,CFDIN,DUCTSTORE.CO.UK,https://www.ductstore.co.uk/cgi-bin/sh000055.p...,March 24 ce price updates prices shown ex vat ...,ductstore.co,Yes,NaN,NaN,NaN,"<!DOCTYPE html PUBLIC ""-//W3C//DTD HTML 4.01 T..."
2,NaN,10537100,10537100,Fusible Link Fire Damper with installation fra...,1,each,82.87,NaN,NaN,NaN,...,CFDIN,DUCTSTORE.CO.UK,https://www.ductstore.co.uk/cgi-bin/sh000055.p...,March 24 ce price updates prices shown ex vat ...,ductstore.co,Yes,NaN,NaN,NaN,"<!DOCTYPE html PUBLIC ""-//W3C//DTD HTML 4.01 T..."
3,NaN,10537102,10537102,Fusible Link Fire Damper with installation fra...,1,each,82.87,NaN,NaN,NaN,...,CFDIN,DUCTSTORE.CO.UK,https://www.ductstore.co.uk/cgi-bin/sh000055.p...,March 24 ce price updates prices shown ex vat ...,ductstore.co,Yes,NaN,NaN,NaN,"<!DOCTYPE html PUBLIC ""-//W3C//DTD HTML 4.01 T..."
4,NaN,45263616,45263616,Straight Rect Steel Ductwork -500 x 500mm,2,m,72.85,NaN,NaN,NaN,...,RDS500-500,DUCTSTORE.CO.UK,https://www.ductstore.co.uk/cgi-bin/sh000055.p...,Slip Joints Mar/Apr 24 update price increase ...,ductstore.co,Yes,NaN,NaN,NaN,"<!DOCTYPE html PUBLIC ""-//W3C//DTD HTML 4.01 T..."


In [84]:
from bs4 import BeautifulSoup
import re

def html_to_text(html: str) -> str:
    if not html:
        return ""

    soup = BeautifulSoup(html, "lxml")

    # Remove non-content elements
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np


def create_html_embeddings(
    df: pd.DataFrame,
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    text_column: str = "html_source",
    embedding_column: str = "embedding"
) -> pd.DataFrame:
    """
    Create embeddings from HTML source and store them in DataFrame.
    """

    if text_column not in df.columns:
        raise ValueError(f"'{text_column}' column not found")

    df = df.copy()

    # Convert HTML → text
    df["clean_text"] = df[text_column].apply(html_to_text)

    model = SentenceTransformer(model_name)

    texts = df["clean_text"].tolist()
    embeddings = model.encode(
        texts,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True  # important for cosine similarity
    )

    df[embedding_column] = list(embeddings)

    return df


In [ ]:
def save_embeddings_pickle(df: pd.DataFrame, path: str):
    df.to_pickle(path)


In [ ]:
embedding_df = create_html_embeddings(df)
save_embeddings_pickle(embedding_df, "page_embeddings_pipelagging.pkl")

In [85]:
embeddings_df = pd.read_pickle("page_embeddings.pkl")


In [ ]:
embeddings_df

In [86]:
def build_prototypes(df, type_col="Type", emb_col="embedding"):
    """
    Returns a dict: {type: embedding}
    """
    prototypes = {}

    for t in sorted(df[type_col].unique()):
        row = df[df[type_col] == t].iloc[0]
        prototypes[t] = row[emb_col]

    return prototypes


In [87]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def predict_type(embedding, prototypes):
    best_type = None
    best_score = -1

    emb = np.array(embedding).reshape(1, -1)

    for t, proto_emb in prototypes.items():
        proto_emb = np.array(proto_emb).reshape(1, -1)
        score = cosine_similarity(emb, proto_emb)[0][0]

        if score > best_score:
            best_score = score
            best_type = t

    return best_type, best_score


In [88]:
def evaluate_prototype_classifier(
    df,
    prototypes,
    type_col="Type",
    emb_col="embedding"
):
    df = df.copy()

    preds = []
    scores = []

    for _, row in df.iterrows():
        pred_type, score = predict_type(row[emb_col], prototypes)
        preds.append(pred_type)
        scores.append(score)

    df["predicted_type"] = preds
    df["similarity_score"] = scores
    df["correct"] = df["predicted_type"] == df[type_col]

    accuracy = df["correct"].mean()

    return df, accuracy


In [89]:
prototypes = build_prototypes(embeddings_df)

results_df, accuracy = evaluate_prototype_classifier(embeddings_df, prototypes)

print(f"Accuracy: {accuracy:.2%}")


Accuracy: 99.07%


In [90]:
results_df[results_df['Type'] != results_df['predicted_type']]

,Unnamed: 0,Code,Code.1,Description,Type,Unit\n,Rate £,Extracted Price,New Rate (ex VAT),New Rate INCL VAT),...,Web,Status,Unnamed: 21,InApp Comment,html_source,clean_text,embedding,predicted_type,similarity_score,correct
58,NaN,45263900,45263900,Duct Rect Steel T Piece - 100 x 100 x 100mm,5,nr,34.09,NaN,NaN,NaN,...,Yes,NaN,NaN,NaN,"<!DOCTYPE html PUBLIC ""-//W3C//DTD HTML 4.01 T...",Ductwork T Pieces 100mm Width 100mm Depth | Re...,"[-0.012264214, 0.024149353, -0.00559127, -0.07...",2,0.913704,False


In [81]:
results_df.describe()

,Unnamed: 0,Type,Rate £,Extracted Price,New Rate (ex VAT),New Rate INCL VAT),% change,Discount,Wastage,Unnamed: 21,predicted_type,similarity_score
count,39.000000,79.000000,79.000000,0.0,17.000000,0.0,17.0,79.0,7.900000e+01,0.0,79.000000,79.000000
mean,20.000000,2.303797,11.392785,NaN,1.525294,NaN,0.0,0.0,5.000000e-02,NaN,2.316456,0.996782
std,11.401754,1.371453,14.896787,NaN,0.246123,NaN,0.0,0.0,2.094970e-17,NaN,1.382531,0.009860
min,1.000000,1.000000,1.170000,NaN,1.170000,NaN,0.0,0.0,5.000000e-02,NaN,1.000000,0.958141
25%,10.500000,1.000000,2.435000,NaN,1.280000,NaN,0.0,0.0,5.000000e-02,NaN,1.000000,1.000000
50%,20.000000,2.000000,7.260000,NaN,1.550000,NaN,0.0,0.0,5.000000e-02,NaN,2.000000,1.000000
75%,29.500000,4.000000,12.915000,NaN,1.690000,NaN,0.0,0.0,5.000000e-02,NaN,4.000000,1.000000
max,39.000000,4.000000,78.100000,NaN,1.940000,NaN,0.0,0.0,5.000000e-02,NaN,4.000000,1.000000


In [91]:
results_df['clean_text'][0]

'Circular Fire Dampers Back Up a Level Home FAQ Gallery Contact Us Our Location Terms & Conds Site Map View Basket Checkout Search the Site Product Sections Spiral Tube Duct & Fittings Duct Fittings with Seals Lindab Safe Duct Fittings Fabricated Round Fittings Stainless Steel Duct Flexible Ducting Grilles & Louvres Supply & Exhaust Valves Round Vents & Wall Cowls Grille & Plenum Boxes Dampers & Access Doors Attenuators (Silencers) Filters & Filter Boxes Flanges & Connectors Fixings & Supports Sealant, Tapes & Insulation Flashings Rectangular Ducting Fume Extraction Hoods Drip Trays Fans & Actuators Clearance Items . New Products Nylon Wall Plugs 8mm - Box of 100 150 dia Plastic Supply/Exhaust Valves 2mm Wire Rope 150m Reel Wires with Carabina 2mm x 2m - Pack of 10 Shield Anchors M8 Ductwrap - Isover ClimCover Alu2 25mm 18m Roll Please note we are currently unable to dispatch to Northern Ireland due to high carriage costs. Please note that we no longer accept Paypal payments. Circular 